## Oresund

In [22]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [23]:
import mikeio
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import modelskill as ms
import torch

# Add parent directory to path to access src module
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent.parent))
print("Current working directory:", Path.cwd())
print("Parent directory added to path:", Path.cwd().parent.parent)

# dataset helper
from helpers import get_altimetry_obs, get_wl_point_obs
from helpers import plot_shred_reconstruction, extract_water_level_measurements, eval_all

# Importing modules from src
from src.datamanager import ROMDataManager
from src.engine import SHREDEngine
from src.train import fit, evaluate
from src.mSHRED.mSHRED import mSHRED

# get device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Current working directory: /home/matteo/Documents/Masked-SHRED/Notebooks_final/ORESUND
Parent directory added to path: /home/matteo/Documents/Masked-SHRED


## Load Data and Measurement

In [24]:
# OPEN FOLDERS
dfs_fldr = "datasets/output/Area.dfsu"
msh_fldr = "datasets/input/mesh_EMOD.mesh"
obs_fldr  = "datasets/observations/"
msh = mikeio.open(msh_fldr)
dfs = mikeio.open(dfs_fldr)
geom = dfs.geometry

ds = dfs.read(time = slice("2020-01-01", "2023-12-31"))

# sensor coverage
alllist = get_wl_point_obs()

# For sparse data simulation
#hf = ["Drogden", "Klagshamn", "Dragor", "Barseback"]
# availabilities 97-50-98-50
#alllist = [s for s in alllist if s.name in hf]

# Extract the values and convert to torch tensors

measurements = extract_water_level_measurements(alllist, ds_time=ds.time, obs_fldr=obs_fldr)
measurements = torch.as_tensor(measurements, dtype=torch.float32)
print("IN:\nMeasurements shape:", measurements.shape)

# Build a single mask for nodes that are valid in all output fields
surface = ds["Surface elevation"].values
u_velocity = ds["U velocity"].values
v_velocity = ds["V velocity"].values
valid_nodes = np.isfinite(surface).all(axis=0) & np.isfinite(u_velocity).all(axis=0) & np.isfinite(v_velocity).all(axis=0)

# outputs
H = torch.from_numpy(surface[:, valid_nodes].astype(np.float32, copy=False))
U = torch.from_numpy(u_velocity[:, valid_nodes].astype(np.float32, copy=False))
V = torch.from_numpy(v_velocity[:, valid_nodes].astype(np.float32, copy=False))

# print the shape of the arrays
print("OUT:\nH shape:", H.shape)
print("U shape:", U.shape)
print("V shape:", V.shape)

# free memory
del surface, u_velocity, v_velocity, valid_nodes, dfs

IN:
Measurements shape: torch.Size([70127, 12])
OUT:
H shape: torch.Size([70127, 3320])
U shape: torch.Size([70127, 3320])
V shape: torch.Size([70127, 3320])


In [25]:
# Assuming 30 days per month, 24 hours per day, and measurements every 30 minutes (2 per hour)
lenmonths = (30 * 24 * 2)  
nmonths = H.shape[0] // lenmonths 
ntime_keeped = nmonths * lenmonths 
print(f"Time keeprd: {ntime_keeped}/{H.shape[0]} total steps ({nmonths} months) -> coverage: {100*ntime_keeped/H.shape[0]:.1f}% \n")

# reshape the arrays to (nmonths, lenmonths, nfeatures) for training
H = H[:ntime_keeped].reshape(nmonths, lenmonths, -1)
U = U[:ntime_keeped].reshape(nmonths, lenmonths, -1)
V = V[:ntime_keeped].reshape(nmonths, lenmonths, -1)
measurements = measurements[:ntime_keeped].reshape(nmonths, lenmonths, -1)  

print("Reshaped H shape:", H.shape)
print("Reshaped U shape:", U.shape)
print("Reshaped V shape:", V.shape)
print("Reshaped measurements shape:", measurements.shape)

#permute the months 
np.random.seed(42)  # For reproducibility
permuted_indices = np.random.permutation(nmonths)
H = H[permuted_indices]
U = U[permuted_indices]
V = V[permuted_indices]
measurements = measurements[permuted_indices]

print("\npermutation indices:", permuted_indices)

Time keeprd: 69120/70127 total steps (48 months) -> coverage: 98.6% 

Reshaped H shape: torch.Size([48, 1440, 3320])
Reshaped U shape: torch.Size([48, 1440, 3320])
Reshaped V shape: torch.Size([48, 1440, 3320])
Reshaped measurements shape: torch.Size([48, 1440, 12])

permutation indices: [27 40 26 43 24 37 12 19  4 25  8  3  6 39 33 13 17 45 15  9 16 29 32 46
  0 31 30  5 11 34  1 44 21  2 36 35 23 41 10 22 18 47 20  7 42 14 28 38]


In [26]:

# ROMDataManager
LAGS = 24

TRAIN_SIZE = 0.70        # percentage of data used for training 
VAL_SIZE = 0.15          # percentage of data used for validation
TEST_SIZE = 0.15         # percentage of data used for testing

COMPRESS_H = 15         # modes kept for H (surface elevation)
COMPRESS_U = 100         # modes kept for U (U velocity)
COMPRESS_V = 100         # modes kept for V (V velocity)

# Training
OPTIMIZER = "AdamW"     # optimizer to use
LEARNING_RATE = 5e-4    # learning rate for training
NUM_EPOCHS = 200        # maximum number of epochs for training
LR_SCHEDULER = "CosineAnnealingLR"      # learning rate scheduler to use
SCHEDULER_STEP_SIZE = NUM_EPOCHS        # step size for learning rate scheduler (if applicable)

BATCH_SIZE = 32         # batch size for training
PATIENCE = 40           # number of epochs with no improvement on validation loss before stopping training
VERBOSE_TRAIN = True

# Architecture
ENCODER_DEPTH = 3               # number of encoder layers
HIDDEN_SIZE = 32                # hidden size for encoder and decoder layers
N_HEADS = 4                     # number of attention heads (if using a transformer-based architecture)
DIM_FEEDFORWARD = 48            # dimension of feedforward network in transformer (if using a transformer-based architecture)
DECODER = "cnn"                 # type of decoder to use (mpl or cnn)   
DECODER_HIDDEN_SIZES = [64,96]     # hidden sizes for decoder layers 
DROPOUT = 0.1                   # dropout rate for encoder and decoder layers
MASKING_TYPE = "mask_data"      # type of masking to apply to inputs during training (mask_data, mask_data_random, etc.)

## (T.1.A): Median imputer

In [ ]:
# Data manager
manager = ROMDataManager(lags=LAGS, train_size=TRAIN_SIZE, val_size=VAL_SIZE, test_size=TEST_SIZE)
# add data to manager
manager.add_data(H, id="Surface elevation", compress=COMPRESS_H, measurements=measurements, imputer="median", verbose=True)
manager.add_data(U, id="U velocity", compress=COMPRESS_U, verbose=True)
manager.add_data(V, id="V velocity", compress=COMPRESS_V, verbose=True)
# prepare datasets
train_dataset, val_dataset, test_dataset = manager.prepare(verbose=True)

# network input and output shapes
nn_input_shape = train_dataset[0][0].shape[1]
nn_output_shape = train_dataset[0][1].shape[0]
print("\nInput shape for the model:", nn_input_shape)
print("Output shape for the model:", nn_output_shape)

In [ ]:
# mSHRED TRAINING
model_args = dict(
    # Encoder
    encoder="transformer", encoder_depth=ENCODER_DEPTH, input_size=nn_input_shape, n_heads=N_HEADS,
    dim_feedforward=DIM_FEEDFORWARD, hidden_size=HIDDEN_SIZE, input_length=LAGS, double_channels=False,
    # Decoder
    decoder=DECODER, decoder_hidden_sizes=DECODER_HIDDEN_SIZES, output_size=nn_output_shape, dropout=DROPOUT,
)
mshred = mSHRED(args=model_args).to(device)

model_path = Path("models_saved_2/T1A.pth")
model_path.parent.mkdir(parents=True, exist_ok=True)

#if not already trained, train the model and save it, otherwise load the model from disk
if not model_path.exists():
    print("Training mSHRED model...")
    train_errors, valid_errors = fit(
        mshred, train_dataset, val_dataset,
        optimizer=OPTIMIZER, lr=LEARNING_RATE, num_epochs=NUM_EPOCHS, patience=PATIENCE, verbose=VERBOSE_TRAIN,
        lr_scheduler=LR_SCHEDULER, scheduler_step_size=SCHEDULER_STEP_SIZE,
    )

    torch.save(mshred.state_dict(), model_path)

else:
    print("Loading pre-trained mSHRED model...")
    mshred.load_state_dict(torch.load(model_path, map_location=device))
    mshred.eval()

eval_all(manager, mshred, train_dataset, val_dataset, test_dataset, H, U, V, model_name="T1A")

del model_args, model_path, mshred
del train_dataset, val_dataset, test_dataset


Loading pre-trained mSHRED model...


RuntimeError: Error(s) in loading state_dict for mSHRED:
	size mismatch for encoder.input_embedding.weight: copying a param with shape torch.Size([32, 13]) from checkpoint, the shape in current model is torch.Size([32, 12]).

## (T.1.B): Mean imputer

In [ ]:
# Data manager
manager = ROMDataManager(lags=LAGS, train_size=TRAIN_SIZE, val_size=VAL_SIZE, test_size=TEST_SIZE)
# add data to manager
manager.add_data(H, id="Surface elevation", compress=COMPRESS_H, measurements=measurements, imputer="mean")
manager.add_data(U, id="U velocity", compress=COMPRESS_U)
manager.add_data(V, id="V velocity", compress=COMPRESS_V)
# prepare datasets
train_dataset, val_dataset, test_dataset = manager.prepare(verbose=True)

# network input and output shapes
nn_input_shape = train_dataset[0][0].shape[1]
nn_output_shape = train_dataset[0][1].shape[0]
print("\nInput shape for the model:", nn_input_shape)
print("Output shape for the model:", nn_output_shape)


Datasets: ['Surface elevation', 'U velocity', 'V velocity']

Number of nan values in sensor measurements after masking: 0

Data shapes after preparation:
Train:	 torch.Size([47520, 24, 12]) -> torch.Size([47520, 215])
Val:	 torch.Size([10080, 24, 12]) -> torch.Size([10080, 215])
Test:	 torch.Size([11520, 24, 12]) -> torch.Size([11520, 215])

Input shape for the model: 12
Output shape for the model: 215


In [ ]:
# mSHRED TRAINING
model_args = dict(
    # Encoder
    encoder="transformer", encoder_depth=ENCODER_DEPTH, input_size=nn_input_shape, n_heads=N_HEADS,
    dim_feedforward=DIM_FEEDFORWARD, hidden_size=HIDDEN_SIZE, input_length=LAGS, double_channels=False,
    # Decoder
    decoder=DECODER, decoder_hidden_sizes=DECODER_HIDDEN_SIZES, output_size=nn_output_shape, dropout=DROPOUT,
)
mshred = mSHRED(args=model_args).to(device)

# Use cosine annealing LR schedule

model_path = Path("models_saved_2/T1B.pth")
model_path.parent.mkdir(parents=True, exist_ok=True)

#if not already trained, train the model and save it, otherwise load the model from disk
if not model_path.exists():
    print("Training mSHRED model...")
    train_errors, valid_errors = fit(
        mshred, train_dataset, val_dataset,
        optimizer=OPTIMIZER, lr=LEARNING_RATE, num_epochs=NUM_EPOCHS, patience=PATIENCE, verbose=VERBOSE_TRAIN,
        lr_scheduler=LR_SCHEDULER, scheduler_step_size=SCHEDULER_STEP_SIZE,
    )

    torch.save(mshred.state_dict(), model_path)

else:
    print("Loading pre-trained mSHRED model...")
    mshred.load_state_dict(torch.load(model_path, map_location=device))
    mshred.eval()

eval_all(manager, mshred, train_dataset, val_dataset, test_dataset, H, U, V, model_name="T1B")

del model_args, model_path, mshred
del train_dataset, val_dataset, test_dataset


Training mSHRED model...
Starting training...


Training:  90%|█████████ | 181/200 [1:09:30<07:17, 23.04s/it, Train Loss=0.00343, Val Loss=0.00436, Patience=39, Best Val Loss=0.00434, LR=9.93e-06]


Early stopping at epoch 182
Training completed with best validation loss: 0.0043


## (T.1.C): Last value imputer

In [ ]:
# Data manager
manager = ROMDataManager(lags=LAGS, train_size=TRAIN_SIZE, val_size=VAL_SIZE, test_size=TEST_SIZE)
# add data to manager
manager.add_data(H, id="Surface elevation", compress=COMPRESS_H, measurements=measurements, imputer="last_value")
manager.add_data(U, id="U velocity", compress=COMPRESS_U)
manager.add_data(V, id="V velocity", compress=COMPRESS_V)
# prepare datasets
train_dataset, val_dataset, test_dataset = manager.prepare(verbose=True)

# network input and output shapes
nn_input_shape = train_dataset[0][0].shape[1]
nn_output_shape = train_dataset[0][1].shape[0]
print("\nInput shape for the model:", nn_input_shape)
print("Output shape for the model:", nn_output_shape)


Datasets: ['Surface elevation', 'U velocity', 'V velocity']

Number of nan values in sensor measurements after masking: 0

Data shapes after preparation:
Train:	 torch.Size([47520, 24, 12]) -> torch.Size([47520, 215])
Val:	 torch.Size([10080, 24, 12]) -> torch.Size([10080, 215])
Test:	 torch.Size([11520, 24, 12]) -> torch.Size([11520, 215])

Input shape for the model: 12
Output shape for the model: 215


In [ ]:
# mSHRED TRAINING
model_args = dict(
    # Encoder
    encoder="transformer", encoder_depth=ENCODER_DEPTH, input_size=nn_input_shape, n_heads=N_HEADS,
    dim_feedforward=DIM_FEEDFORWARD, hidden_size=HIDDEN_SIZE, input_length=LAGS, double_channels=False,
    # Decoder
    decoder=DECODER, decoder_hidden_sizes=DECODER_HIDDEN_SIZES, output_size=nn_output_shape, dropout=DROPOUT,
)
mshred = mSHRED(args=model_args).to(device)

# Use cosine annealing LR schedule

model_path = Path("models_saved_2/T1C.pth")
model_path.parent.mkdir(parents=True, exist_ok=True)

#if not already trained, train the model and save it, otherwise load the model from disk
if not model_path.exists():
    print("Training mSHRED model...")
    train_errors, valid_errors = fit(
        mshred, train_dataset, val_dataset,
        optimizer=OPTIMIZER, lr=LEARNING_RATE, num_epochs=NUM_EPOCHS, patience=PATIENCE, verbose=VERBOSE_TRAIN,
        lr_scheduler=LR_SCHEDULER, scheduler_step_size=SCHEDULER_STEP_SIZE,
    )

    torch.save(mshred.state_dict(), model_path)

else:
    print("Loading pre-trained mSHRED model...")
    mshred.load_state_dict(torch.load(model_path, map_location=device))
    mshred.eval()

eval_all(manager, mshred, train_dataset, val_dataset, test_dataset, H, U, V, model_name="T1C")

del model_args, model_path, mshred
del train_dataset, val_dataset, test_dataset

Training mSHRED model...
Starting training...


Training: 100%|██████████| 200/200 [1:07:45<00:00, 20.33s/it, Train Loss=0.00328, Val Loss=0.00423, Patience=31, Best Val Loss=0.00423, LR=0.00e+00]


Training completed with best validation loss: 0.0042


## (T.2.A) GRUD

In [ ]:
# Data manager
manager = ROMDataManager(lags=LAGS, train_size=TRAIN_SIZE, val_size=VAL_SIZE, test_size=TEST_SIZE)
# add data to manager
manager.add_data(H, id="Surface elevation", compress=COMPRESS_H, measurements=measurements)
manager.add_data(U, id="U velocity", compress=COMPRESS_U)
manager.add_data(V, id="V velocity", compress=COMPRESS_V)
# prepare datasets
train_dataset, val_dataset, test_dataset, X_mean = manager.prepare(verbose=True, masking_type="grud")

# network input and output shapes
nn_input_shape = train_dataset[0][0].shape[-1]
nn_output_shape = train_dataset[0][1].shape[0]
print("\nInput shape for the model:", nn_input_shape)
print("Output shape for the model:", nn_output_shape)


Datasets: ['Surface elevation', 'U velocity', 'V velocity']

Number of nan values in sensor measurements after masking: 0

Data shapes after preparation:
Train:	 torch.Size([47520, 4, 24, 12]) -> torch.Size([47520, 215])
Val:	 torch.Size([10080, 4, 24, 12]) -> torch.Size([10080, 215])
Test:	 torch.Size([11520, 4, 24, 12]) -> torch.Size([11520, 215])

Input shape for the model: 12
Output shape for the model: 215


In [ ]:
# GRUD TRAINING
model_args = dict(
    # Encoder
    encoder="grud", input_size=nn_input_shape, hidden_size=HIDDEN_SIZE, X_mean=X_mean,
    # Decoder
    decoder=DECODER, decoder_hidden_sizes=DECODER_HIDDEN_SIZES, output_size=nn_output_shape, dropout=DROPOUT,
)
mshred = mSHRED(args=model_args).to(device)

# Use cosine annealing LR schedule

model_path = Path("models_saved_2/T2A.pth")
model_path.parent.mkdir(parents=True, exist_ok=True)

#if not already trained, train the model and save it, otherwise load the model from disk
if not model_path.exists():
    print("Training mSHRED model...")
    train_errors, valid_errors = fit(
        mshred, train_dataset, val_dataset,
        optimizer=OPTIMIZER, lr=LEARNING_RATE, num_epochs=NUM_EPOCHS, patience=PATIENCE, verbose=VERBOSE_TRAIN,
        lr_scheduler=LR_SCHEDULER, scheduler_step_size=SCHEDULER_STEP_SIZE,
    )

    torch.save(mshred.state_dict(), model_path)

else:
    print("Loading pre-trained mSHRED model...")
    mshred.load_state_dict(torch.load(model_path, map_location=device))
    mshred.eval()

eval_all(manager, mshred, train_dataset, val_dataset, test_dataset, H, U, V, model_name="T2A")

del model_args, model_path, mshred


Training mSHRED model...
Starting training...


Training:  98%|█████████▊| 195/200 [38:31<00:59, 11.85s/it, Train Loss=0.00338, Val Loss=0.00428, Patience=39, Best Val Loss=0.00427, LR=4.93e-07]


Early stopping at epoch 196
Training completed with best validation loss: 0.0043


## (T.2.B): Attn-GRUD

In [ ]:
# Attn-GRUD TRAINING
model_args = dict(
    # Encoder
    encoder="att_grud", input_size=nn_input_shape, hidden_size=HIDDEN_SIZE, X_mean=X_mean,
    # Decoder
    decoder=DECODER, decoder_hidden_sizes=DECODER_HIDDEN_SIZES, output_size=nn_output_shape, dropout=DROPOUT,
)
mshred = mSHRED(args=model_args).to(device)

# Use cosine annealing LR schedule

model_path = Path("models_saved_2/T2B.pth")
model_path.parent.mkdir(parents=True, exist_ok=True)

#if not already trained, train the model and save it, otherwise load the model from disk
if not model_path.exists():
    print("Training mSHRED model...")
    train_errors, valid_errors = fit(
        mshred, train_dataset, val_dataset,
        optimizer=OPTIMIZER, lr=LEARNING_RATE, num_epochs=NUM_EPOCHS, patience=PATIENCE, verbose=VERBOSE_TRAIN,
        lr_scheduler=LR_SCHEDULER, scheduler_step_size=SCHEDULER_STEP_SIZE,
    )

    torch.save(mshred.state_dict(), model_path)

else:
    print("Loading pre-trained mSHRED model...")
    mshred.load_state_dict(torch.load(model_path, map_location=device))
    mshred.eval()

eval_all(manager, mshred, train_dataset, val_dataset, test_dataset, H, U, V, model_name="T2B")

del model_args, model_path, mshred
del train_dataset, val_dataset, test_dataset

Training mSHRED model...
Starting training...


Training:  86%|████████▌ | 171/200 [1:40:57<17:07, 35.42s/it, Train Loss=0.00337, Val Loss=0.00438, Patience=39, Best Val Loss=0.00435, LR=2.38e-05]


Early stopping at epoch 172
Training completed with best validation loss: 0.0044


## (T.3.A): Masking

In [ ]:
# Data manager
manager = ROMDataManager(lags=LAGS, train_size=TRAIN_SIZE, val_size=VAL_SIZE, test_size=TEST_SIZE)
# add data to manager
manager.add_data(H, id="Surface elevation", compress=COMPRESS_H, measurements=measurements)
manager.add_data(U, id="U velocity", compress=COMPRESS_U)
manager.add_data(V, id="V velocity", compress=COMPRESS_V)
# prepare datasets
train_dataset, val_dataset, test_dataset = manager.prepare(verbose=True, masking_type=MASKING_TYPE)

# network input and output shapes
nn_input_shape = train_dataset.X.shape[-1]
nn_output_shape = train_dataset.Y.shape[-1]
print("\nInput shape for the model:", nn_input_shape)
print("Output shape for the model:", nn_output_shape)


Datasets: ['Surface elevation', 'U velocity', 'V velocity']

Number of nan values in sensor measurements after masking: 233523

Data shapes after preparation:
Train:	 torch.Size([47520, 24, 12]) -> torch.Size([47520, 215])
Val:	 torch.Size([10080, 24, 12]) -> torch.Size([10080, 215])
Test:	 torch.Size([11520, 24, 12]) -> torch.Size([11520, 215])

Input shape for the model: 12
Output shape for the model: 215


In [ ]:
# mSHRED TRAINING
model_args = dict(
    # Encoder
    encoder="transformer", encoder_depth=ENCODER_DEPTH, input_size=nn_input_shape, n_heads=N_HEADS,
    dim_feedforward=DIM_FEEDFORWARD, hidden_size=HIDDEN_SIZE, input_length=LAGS, double_channels=False,
    # Decoder
    decoder=DECODER, decoder_hidden_sizes=DECODER_HIDDEN_SIZES, output_size=nn_output_shape, dropout=DROPOUT, 
)
mshred = mSHRED(args=model_args).to(device)

# Use cosine annealing LR schedule

model_path = Path("models_saved_2/T3A.pth")
model_path.parent.mkdir(parents=True, exist_ok=True)

#if not already trained, train the model and save it, otherwise load the model from disk
if not model_path.exists():
    print("Training mSHRED model...")
    train_errors, valid_errors = fit(
        mshred, train_dataset, val_dataset,
        optimizer=OPTIMIZER, lr=LEARNING_RATE, num_epochs=NUM_EPOCHS, patience=PATIENCE, verbose=VERBOSE_TRAIN,
        lr_scheduler=LR_SCHEDULER, scheduler_step_size=SCHEDULER_STEP_SIZE,
        masking_type=MASKING_TYPE,
    )

    torch.save(mshred.state_dict(), model_path)

else:
    print("Loading pre-trained mSHRED model...")
    mshred.load_state_dict(torch.load(model_path, map_location=device))
    mshred.eval()

eval_all(manager, mshred, train_dataset, val_dataset, test_dataset, H, U, V, model_name="T3A")

del model_args, model_path, mshred


Training mSHRED model...
Starting training...


Training: 100%|██████████| 200/200 [1:20:46<00:00, 24.23s/it, Train Loss=0.00343, Val Loss=0.00437, Patience=13, Best Val Loss=0.00437, LR=0.00e+00]


Training completed with best validation loss: 0.0044


## (T.3.B): Masking x2 channels

In [ ]:
# mSHRED TRAINING
model_args = dict(
    # Encoder
    encoder="transformer", encoder_depth=ENCODER_DEPTH, input_size=nn_input_shape, n_heads=N_HEADS,
    dim_feedforward=DIM_FEEDFORWARD, hidden_size=HIDDEN_SIZE, input_length=LAGS, double_channels=True,
    # Decoder
    decoder=DECODER, decoder_hidden_sizes=DECODER_HIDDEN_SIZES, output_size=nn_output_shape, dropout=DROPOUT, 
 )
mshred = mSHRED(args=model_args).to(device)

# Use cosine annealing LR schedule

model_path = Path("models_saved_2/T3B.pth")
model_path.parent.mkdir(parents=True, exist_ok=True)

#if not already trained, train the model and save it, otherwise load the model from disk
if not model_path.exists():
    print("Training mSHRED model...")
    train_errors, valid_errors = fit(
        mshred, train_dataset, val_dataset,
        optimizer=OPTIMIZER, lr=LEARNING_RATE, num_epochs=NUM_EPOCHS, patience=PATIENCE, verbose=VERBOSE_TRAIN,
        lr_scheduler=LR_SCHEDULER, scheduler_step_size=SCHEDULER_STEP_SIZE,
        masking_type=MASKING_TYPE,
     )

    torch.save(mshred.state_dict(), model_path)

else:
    print("Loading pre-trained mSHRED model...")
    mshred.load_state_dict(torch.load(model_path, map_location=device))
    mshred.eval()

eval_all(manager, mshred, train_dataset, val_dataset, test_dataset, H, U, V, model_name="T3B")

del model_args, model_path, mshred


Training mSHRED model...
Starting training...


Training:  96%|█████████▌| 191/200 [1:16:38<03:36, 24.07s/it, Train Loss=0.00337, Val Loss=0.00433, Patience=39, Best Val Loss=0.00431, LR=1.97e-06]


Early stopping at epoch 192
Training completed with best validation loss: 0.0043


## (T.3.C): Masking hidden space

In [ ]:
# mSHRED TRAINING
model_args = dict(
    # Encoder
    encoder="transformer_hidden_mask", encoder_depth=ENCODER_DEPTH, input_size=nn_input_shape, n_heads=N_HEADS,
    dim_feedforward=DIM_FEEDFORWARD, hidden_size=HIDDEN_SIZE, input_length=LAGS,
    # Decoder
    decoder=DECODER, decoder_hidden_sizes=DECODER_HIDDEN_SIZES, output_size=nn_output_shape, dropout=DROPOUT, 
 )
mshred = mSHRED(args=model_args).to(device)

# Use cosine annealing LR schedule

model_path = Path("models_saved_2/T3C.pth")
model_path.parent.mkdir(parents=True, exist_ok=True)

#if not already trained, train the model and save it, otherwise load the model from disk
if not model_path.exists():
    print("Training mSHRED model...")
    train_errors, valid_errors = fit(
        mshred, train_dataset, val_dataset,
        optimizer=OPTIMIZER, lr=LEARNING_RATE, num_epochs=NUM_EPOCHS, patience=PATIENCE, verbose=VERBOSE_TRAIN,
        lr_scheduler=LR_SCHEDULER, scheduler_step_size=SCHEDULER_STEP_SIZE,
        masking_type=MASKING_TYPE,
     )

    torch.save(mshred.state_dict(), model_path)

else:
    print("Loading pre-trained mSHRED model...")
    mshred.load_state_dict(torch.load(model_path, map_location=device))
    mshred.eval()

eval_all(manager, mshred, train_dataset, val_dataset, test_dataset, H, U, V, model_name="T3C")

del model_args, model_path, mshred


Training mSHRED model...
Starting training...


Training: 100%|██████████| 200/200 [1:19:13<00:00, 23.77s/it, Train Loss=0.00339, Val Loss=0.00427, Patience=22, Best Val Loss=0.00427, LR=0.00e+00]


Training completed with best validation loss: 0.0043


## (T.3.D): Masking MHSA

In [ ]:
# mSHRED TRAINING
model_args = dict(
    # Encoder
    encoder="transformer_attn_mask", encoder_depth=ENCODER_DEPTH, input_size=nn_input_shape, n_heads=N_HEADS,
    dim_feedforward=DIM_FEEDFORWARD, hidden_size=HIDDEN_SIZE, input_length=LAGS, double_channels=False,
    # Decoder
    decoder=DECODER, decoder_hidden_sizes=DECODER_HIDDEN_SIZES, output_size=nn_output_shape, dropout=DROPOUT, 
 )
mshred = mSHRED(args=model_args).to(device)

# Use cosine annealing LR schedule

model_path = Path("models_saved_2/T3D.pth")
model_path.parent.mkdir(parents=True, exist_ok=True)

#if not already trained, train the model and save it, otherwise load the model from disk
if not model_path.exists():
    print("Training mSHRED model...")
    train_errors, valid_errors = fit(
        mshred, train_dataset, val_dataset,
        optimizer=OPTIMIZER, lr=LEARNING_RATE, num_epochs=NUM_EPOCHS, patience=PATIENCE, verbose=VERBOSE_TRAIN,
        lr_scheduler=LR_SCHEDULER, scheduler_step_size=SCHEDULER_STEP_SIZE,
        masking_type=MASKING_TYPE,
     )

    torch.save(mshred.state_dict(), model_path)

else:
    print("Loading pre-trained mSHRED model...")
    mshred.load_state_dict(torch.load(model_path, map_location=device))
    mshred.eval()

eval_all(manager, mshred, train_dataset, val_dataset, test_dataset, H, U, V, model_name="T3D")

del model_args, model_path, mshred


Training mSHRED model...
Starting training...


Training: 100%|██████████| 200/200 [1:24:49<00:00, 25.45s/it, Train Loss=0.00349, Val Loss=0.00436, Patience=32, Best Val Loss=0.00435, LR=0.00e+00]


Training completed with best validation loss: 0.0043


## (T.3.E): Masking x2 channels + MHSA

In [ ]:
# mSHRED TRAINING
model_args = dict(
    # Encoder
    encoder="transformer_attn_mask", encoder_depth=ENCODER_DEPTH, input_size=nn_input_shape, n_heads=N_HEADS,
    dim_feedforward=DIM_FEEDFORWARD, hidden_size=HIDDEN_SIZE, input_length=LAGS, double_channels=True,
    # Decoder
    decoder=DECODER, decoder_hidden_sizes=DECODER_HIDDEN_SIZES, output_size=nn_output_shape, dropout=DROPOUT, 
 )
mshred = mSHRED(args=model_args).to(device)

# Use cosine annealing LR schedule

model_path = Path("models_saved_2/T3E.pth")
model_path.parent.mkdir(parents=True, exist_ok=True)

#if not already trained, train the model and save it, otherwise load the model from disk
if not model_path.exists():
    print("Training mSHRED model...")
    train_errors, valid_errors = fit(
        mshred, train_dataset, val_dataset,
        optimizer=OPTIMIZER, lr=LEARNING_RATE, num_epochs=NUM_EPOCHS, patience=PATIENCE, verbose=VERBOSE_TRAIN,
        lr_scheduler=LR_SCHEDULER, scheduler_step_size=SCHEDULER_STEP_SIZE,
        masking_type=MASKING_TYPE,
     )

    torch.save(mshred.state_dict(), model_path)

else:
    print("Loading pre-trained mSHRED model...")
    mshred.load_state_dict(torch.load(model_path, map_location=device))
    mshred.eval()

eval_all(manager, mshred, train_dataset, val_dataset, test_dataset, H, U, V, model_name="T3E")

del model_args, model_path, mshred


Training mSHRED model...
Starting training...


Training:  84%|████████▍ | 169/200 [1:05:45<12:03, 23.35s/it, Train Loss=0.00337, Val Loss=0.00429, Patience=39, Best Val Loss=0.00429, LR=2.72e-05]


Early stopping at epoch 170
Training completed with best validation loss: 0.0043


## (T.3.F): Masking MHSA learned

In [ ]:
# mSHRED TRAINING
model_args = dict(
    # Encoder
    encoder="transformer_learned_mask", encoder_depth=ENCODER_DEPTH, input_size=nn_input_shape, n_heads=N_HEADS,
    dim_feedforward=DIM_FEEDFORWARD, hidden_size=HIDDEN_SIZE, input_length=LAGS,
    # Decoder
    decoder=DECODER, decoder_hidden_sizes=DECODER_HIDDEN_SIZES, output_size=nn_output_shape, dropout=DROPOUT, 
 )
mshred = mSHRED(args=model_args).to(device)

# Use cosine annealing LR schedule

model_path = Path("models_saved_2/T3F.pth")
model_path.parent.mkdir(parents=True, exist_ok=True)

#if not already trained, train the model and save it, otherwise load the model from disk
if not model_path.exists():
    print("Training mSHRED model...")
    train_errors, valid_errors = fit(
        mshred, train_dataset, val_dataset,
        optimizer=OPTIMIZER, lr=LEARNING_RATE, num_epochs=NUM_EPOCHS, patience=PATIENCE, verbose=VERBOSE_TRAIN,
        lr_scheduler=LR_SCHEDULER, scheduler_step_size=SCHEDULER_STEP_SIZE,
        masking_type=MASKING_TYPE,
     )

    torch.save(mshred.state_dict(), model_path)

else:
    print("Loading pre-trained mSHRED model...")
    mshred.load_state_dict(torch.load(model_path, map_location=device))
    mshred.eval()

eval_all(manager, mshred, train_dataset, val_dataset, test_dataset, H, U, V, model_name="T3F")

del model_args, model_path, mshred


Training mSHRED model...
Starting training...


Training:  83%|████████▎ | 166/200 [1:31:22<18:43, 33.03s/it, Train Loss=0.00341, Val Loss=0.00430, Patience=39, Best Val Loss=0.00428, LR=3.28e-05] 


Early stopping at epoch 167
Training completed with best validation loss: 0.0043


## (T.3.G): Masking x2 channels + MHSA learned

In [ ]:
# mSHRED TRAINING
model_args = dict(
    # Encoder
    encoder="transformer_learned_mask", encoder_depth=ENCODER_DEPTH, input_size=nn_input_shape, n_heads=N_HEADS,
    dim_feedforward=DIM_FEEDFORWARD, hidden_size=HIDDEN_SIZE, input_length=LAGS, double_channels=True,
    # Decoder
    decoder=DECODER, decoder_hidden_sizes=DECODER_HIDDEN_SIZES, output_size=nn_output_shape, dropout=DROPOUT, 
 )
mshred = mSHRED(args=model_args).to(device)

# Use cosine annealing LR schedule

model_path = Path("models_saved_2/T3G.pth")
model_path.parent.mkdir(parents=True, exist_ok=True)

#if not already trained, train the model and save it, otherwise load the model from disk
if not model_path.exists():
    print("Training mSHRED model...")
    train_errors, valid_errors = fit(
        mshred, train_dataset, val_dataset,
        optimizer=OPTIMIZER, lr=LEARNING_RATE, num_epochs=NUM_EPOCHS, patience=PATIENCE, verbose=VERBOSE_TRAIN,
        lr_scheduler=LR_SCHEDULER, scheduler_step_size=SCHEDULER_STEP_SIZE,
        masking_type=MASKING_TYPE,
     )

    torch.save(mshred.state_dict(), model_path)

else:
    print("Loading pre-trained mSHRED model...")
    mshred.load_state_dict(torch.load(model_path, map_location=device))
    mshred.eval()

eval_all(manager, mshred, train_dataset, val_dataset, test_dataset, H, U, V, model_name="T3G")

del model_args, model_path, mshred


Training mSHRED model...
Starting training...


Training:  95%|█████████▌| 190/200 [1:23:19<04:23, 26.31s/it, Train Loss=0.00339, Val Loss=0.00432, Patience=39, Best Val Loss=0.00430, LR=2.49e-06]


Early stopping at epoch 191
Training completed with best validation loss: 0.0043
